In [0]:
%sql
-- -- Création des différentes tables

-- -- 2.1 contracts
-- CREATE TABLE IF NOT EXISTS alpenia_dev.bronze.brz_contracts (
--   contract_id STRING COMMENT 'Identifiant unique du contrat (CT-00001)',
--   client_id STRING COMMENT 'Souscripteur',
--   product STRING COMMENT 'auto ou habitation',
--   city STRING COMMENT 'Ville de rattachement',
--   start_date DATE COMMENT 'Date d''effet (fournie au format dd/MM/yyyy)',
--   annual_premium_eur DOUBLE COMMENT 'Prime annuelle',
--   status STRING COMMENT 'active, suspended ou terminated'
-- )
-- USING DELTA;

-- --2.2 clients
-- CREATE TABLE IF NOT EXISTS alpenia_dev.bronze.brz_clients (
--     client_id STRING COMMENT 'Identifiant unique du client',
--     signup_date DATE COMMENT 'Date d\'entrée en portefeuille',
--     city STRING COMMENT 'Ville principale',
--     segment STRING COMMENT 'particulier ou professionnel',
--     birth_year INT COMMENT 'Année de naissance'
-- )
-- USING DELTA;

-- -- 2.3 pricing
-- CREATE TABLE IF NOT EXISTS alpenia_dev.bronze.brz_pricing (
--   city STRING COMMENT 'Ville',
--   product STRING COMMENT 'Produit d''assurance',
--   base_premium_eur DECIMAL(7,2) COMMENT 'Prime de base annuelle',
--   rate_per_1k_eur DECIMAL(5,2) COMMENT 'Taux appliqué par tranche de 1 000 € de capital assuré',
--   valid_from DATE COMMENT 'Date d''entrée en vigueur'
-- ) USING DELTA;

-- -- 2.4 claims
-- CREATE TABLE IF NOT EXISTS alpenia_dev.bronze.brz_claims (
--   claim_id STRING COMMENT 'Identifiant unique du sinistre',
--   contract_id STRING COMMENT 'Contrat concerné',
--   client_id STRING COMMENT 'Client',
--   declared_ts TIMESTAMP COMMENT 'Déclaration du sinistre (UTC)',
--   settled_ts TIMESTAMP COMMENT 'Règlement (UTC)',
--   peril STRING COMMENT 'Nature : degat_des_eaux, bris_de_glace, incendie, vol, collision, catastrophe_naturelle',
--   amount_eur DOUBLE COMMENT 'Montant réglé',
--   deductible_eur DOUBLE COMMENT 'Franchise appliquée'
-- ) USING DELTA;

-- -- 2.5 web_events
-- CREATE TABLE IF NOT EXISTS alpenia_dev.bronze.brz_web_events (
--   event_id STRING COMMENT 'Identifiant unique de l''évènement',
--   client_id STRING COMMENT 'Client identifié sur le parcours',
--   event_ts TIMESTAMP COMMENT 'Horodatage (fourni en texte ISO 8601)',
--   quote STRUCT<quote_product: STRING, quote_premium_eur: DOUBLE> COMMENT 'Devis en cours — à aplatir en quote_product / quote_premium_eur',
--   channel STRING COMMENT 'web ou mobile',
--   action STRING COMMENT 'page_view, quote_started, quote_completed, quote_abandoned ou claim_declared'
-- ) USING DELTA;

-- -- 2.6 expertises
-- CREATE TABLE IF NOT EXISTS alpenia_dev.bronze.brz_expertises (
--   expertise_id STRING COMMENT 'Identifiant unique de l''expertise',
--   claim_id STRING COMMENT 'Sinistre concerné',
--   opened_at TIMESTAMP COMMENT 'Ouverture du dossier d''expertise',
--   closed_at TIMESTAMP COMMENT 'Clôture — NULL si l''expertise est en cours',
--   finding_type STRING COMMENT 'Catégorie de constat',
--   items_replaced ARRAY<STRING> COMMENT 'Postes indemnisés (peut être vide)',
--   cost_eur DOUBLE COMMENT 'Coût de l''expertise'
-- ) USING DELTA;


In [0]:
# Ingestion
BASE = "/Volumes/alpenia_dev/bronze/landing"

# --- CSV Files ---
df_contracts = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("sep", ";") \
    .load(f"{BASE}/contracts/")

df_clients = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .load(f"{BASE}/clients/")

df_pricing = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .load(f"{BASE}/pricing/")

# --- Parquet Files ---

df_claims = spark.read \
    .format("parquet") \
    .load(f"{BASE}/claims/")

# --- JSON Files ---
df_web_events = spark.read \
    .format("json") \
    .load(f"{BASE}/web_events/")

df_expertises = spark.read \
    .format("json") \
    .option("multiLine", "true") \
    .load(f"{BASE}/expertises/")

# Optional: Verify the first few rows of each
display(df_contracts.limit(5))
display(df_clients.limit(5))
display(df_pricing.limit(5))
display(df_claims.limit(5))
display(df_web_events.limit(5))
display(df_expertises.limit(5))


In [0]:
# Ecrire les tables
# --- Write CSV Data to Bronze Tables ---

# 1. Contracts
df_contracts.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("alpenia_dev.bronze.brz_contracts")

# 2. Clients
df_clients.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("alpenia_dev.bronze.brz_clients")

# 3. Pricing
df_pricing.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("alpenia_dev.bronze.brz_pricing")

# --- Write Parquet/JSON Data to Bronze Tables ---

# 4. Claims
df_claims.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("alpenia_dev.bronze.brz_claims")

# 5. Web Events
df_web_events.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("alpenia_dev.bronze.brz_web_events")

# 6. Expertises
df_expertises.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("alpenia_dev.bronze.brz_expertises")



In [0]:
%sql
--Pour se  conformer aux exigences de l'exercice, j'utiliserai en dur les valeurs données dans le 2-
SELECT count(*) AS Actual, 4000 AS Expected, IF(Actual = Expected, 'OK', 'KO') AS Status FROM alpenia_dev.bronze.brz_contracts
UNION ALL
SELECT count(*) AS Actual, 3000 AS Expected, IF(Actual = Expected, 'OK', 'KO') AS Status FROM alpenia_dev.bronze.brz_clients
UNION ALL
SELECT count(*) AS Actual, 6 AS Expected, IF(Actual = Expected, 'OK', 'KO') AS Status FROM alpenia_dev.bronze.brz_pricing
 UNION ALL
SELECT count(*) AS Actual, 24000 AS Expected, IF(Actual = Expected, 'OK', 'KO') AS Status FROM alpenia_dev.bronze.brz_claims
 UNION ALL
SELECT count(*) AS Actual, 90000 AS Expected, IF(Actual = Expected, 'OK', 'KO') AS Status FROM alpenia_dev.bronze.brz_web_events
 UNION ALL
SELECT count(*) AS Actual, 450 AS Expected, IF(Actual = Expected, 'OK', 'KO') AS Status FROM alpenia_dev.bronze.brz_expertises
;
